# 04 — Desafío: preguntas → txt → LLM → csv

## Enunciado
1. Crear un archivo `.txt` desde una lista de preguntas en Python
2. Leer las preguntas del `.txt` a otra lista
3. Obtener respuestas de un LLM para cada pregunta
4. Guardar resultados en `.csv`
5. Leer el `.csv` con Pandas

## Conceptos
- Persistencia: lista → archivo → lista
- Integración LLM + archivos + DataFrame
- Pipeline de datos de punta a punta

## Salidas generadas
- `preguntas.txt`, `respuestas1.csv`, `respuesta2.csv` (en carpeta `output/`)


In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from notebooks._setup import (
    OUTPUT_DIR,
    LocalGeminiClient,
    load_env,
    require_key,
    safe_generate,
    save_question_answers,
    use_real_apis,
)

load_env()
OUTPUT_DIR.mkdir(exist_ok=True)

if use_real_apis():
    require_key("GEMINI_API_KEY")
    from google import genai
    client = genai.Client()
else:
    client = LocalGeminiClient()

Crear un archivo .txt a partir de una lista de preguntas definida en Python.
Leer las preguntas de ese archivo .txt y guardarlas en una lista de Python.
Obtener respuestas de un LLM para cada pregunta.
Guardar los resultados en un nuevo archivo .csv.
Leer el archivo .csv usando Pandas.

In [2]:
# Paso 1: definir preguntas claras y guardarlas en un TXT
lista_de_preguntas = [
    "De que esta hecho el Sol?",
    "De que esta hecho el planeta Saturno?",
    "Cual es la galaxia mas antigua encontrada hasta ahora?",
    "Cual es la estrella mas grande conocida?",
    "Cual es la estrella mas cercana al Sol?",
]

ruta_preguntas = OUTPUT_DIR / "preguntas.txt"
with ruta_preguntas.open("w", encoding="utf-8") as archivo:
    for pregunta in lista_de_preguntas:
        archivo.write(pregunta + "\n")

print(f"Preguntas guardadas en: {ruta_preguntas}")

Preguntas guardadas en: C:\Users\ben14\Downloads\IA_Aplicada\output\preguntas.txt


In [3]:
# Paso 2: leer las preguntas desde el TXT
with (OUTPUT_DIR / "preguntas.txt").open("r", encoding="utf-8") as archivo:
    lista_desafio = [linea.strip() for linea in archivo if linea.strip()]

pd.DataFrame({"pregunta": lista_desafio})

,pregunta
0,De que esta hecho el Sol?
1,De que esta hecho el planeta Saturno?
2,Cual es la galaxia mas antigua encontrada hast...
3,Cual es la estrella mas grande conocida?
4,Cual es la estrella mas cercana al Sol?


In [4]:
# Paso 3: responder cada pregunta y guardar una estructura limpia
lista_respuestas = []
for numero, pregunta in enumerate(lista_desafio, start=1):
    respuesta = safe_generate(
        lambda texto: client.models.generate_content(model="gemini-2.5-flash", contents=texto),
        pregunta,
    )
    lista_respuestas.append({
        "numero": numero,
        "pregunta": pregunta,
        "respuesta": respuesta,
    })

save_question_answers(OUTPUT_DIR / "respuestas_preguntas.csv", lista_respuestas)

df_preguntas_y_respuestas = pd.DataFrame(lista_respuestas)
df_preguntas_y_respuestas

,numero,pregunta,respuesta
0,1,De que esta hecho el Sol?,El Sol esta compuesto principalmente por hidro...
1,2,De que esta hecho el planeta Saturno?,Saturno esta compuesto principalmente por hidr...
2,3,Cual es la galaxia mas antigua encontrada hast...,Las galaxias mas antiguas observadas pertenece...
3,4,Cual es la estrella mas grande conocida?,Una de las estrellas mas grandes conocidas es ...
4,5,Cual es la estrella mas cercana al Sol?,La estrella mas cercana al Sol es Proxima Cent...


In [5]:
# Paso 4: guardar tambien una copia compatible con el ejercicio
save_question_answers(OUTPUT_DIR / "respuestas1.csv", lista_respuestas)
print("Respuestas guardadas en output/respuestas1.csv y output/respuestas_preguntas.csv")

Respuestas guardadas en output/respuestas1.csv y output/respuestas_preguntas.csv


In [6]:
df_preguntas_y_respuestas = pd.DataFrame(lista_respuestas)

In [7]:
df_preguntas_y_respuestas

,numero,pregunta,respuesta
0,1,De que esta hecho el Sol?,El Sol esta compuesto principalmente por hidro...
1,2,De que esta hecho el planeta Saturno?,Saturno esta compuesto principalmente por hidr...
2,3,Cual es la galaxia mas antigua encontrada hast...,Las galaxias mas antiguas observadas pertenece...
3,4,Cual es la estrella mas grande conocida?,Una de las estrellas mas grandes conocidas es ...
4,5,Cual es la estrella mas cercana al Sol?,La estrella mas cercana al Sol es Proxima Cent...


In [8]:
df_preguntas_y_respuestas.to_csv(OUTPUT_DIR / "respuestas2.csv", index=False, encoding="utf-8")

In [9]:
# Paso 5: leer el CSV final con Pandas
nuevo_df = pd.read_csv(OUTPUT_DIR / "respuestas2.csv")

In [10]:
nuevo_df

,numero,pregunta,respuesta
0,1,De que esta hecho el Sol?,El Sol esta compuesto principalmente por hidro...
1,2,De que esta hecho el planeta Saturno?,Saturno esta compuesto principalmente por hidr...
2,3,Cual es la galaxia mas antigua encontrada hast...,Las galaxias mas antiguas observadas pertenece...
3,4,Cual es la estrella mas grande conocida?,Una de las estrellas mas grandes conocidas es ...
4,5,Cual es la estrella mas cercana al Sol?,La estrella mas cercana al Sol es Proxima Cent...


Desafio:

1. Cargar un archivo .csv con feedback de clientes y crear un dataframe
2. Usar un LLM para clasificar el sentimiento de cada feedback (positivo, negativo, neutro)
3. Agregar esa clasificación al DataFrame